# TTA trên E4 — dùng lại 5 checkpoint đã có, KHÔNG train lại

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

Test-time augmentation bằng phép **lật**: chạy inference trên 8 tổ hợp lật rồi trung
bình xác suất. Vài phút GPU cho cả 5 fold.

## Vì sao lật là phép TTA hợp lệ ở đây, còn xoay thì không

TTA chỉ đúng khi model *đáng lẽ* bất biến với phép biến đổi đó. Với lật, đó là sự thật
của chính quá trình train: `flip_prob: 0.5` trên cả ba trục. Với `rot90` thì không —
gan bên phải, lách bên trái, cột sống phía sau; model chưa từng được dạy bất biến với
nó (`rot90_prob: 0`, có chủ ý). Chi tiết ở `src/eval/tta.py`.

## E4 là gì ở đây

Config train là `baseline_3dpatch.yaml` (không sửa gì). Cái làm nên E4 nằm ở **cache**:
cắt bám tổn thương, 112×112×32, căn pha **từng thì**. Cổng A kiểm đúng ba khoá đó.

## Cách đọc kết quả (chốt TRƯỚC khi chạy)

Lượt thứ 0 trong 8 lượt là **ảnh gốc** (tổ hợp lật rỗng), nên nó chính là bản không
TTA. Không cần chạy thêm gì để có đối chứng.

| quan sát | kết luận |
|---|---|
| lượt 0 khớp macro-F1 đã lưu trong checkpoint | đường chạy đúng, số TTA tin được |
| lượt 0 **lệch** | dừng lại: sai cache, sai checkpoint, hoặc model còn ở chế độ train |
| TTA − lượt 0 dương đều ở 5 fold | TTA có tác dụng, đưa vào cấu hình khoá cho test-104 |
| lệch lung tung quanh 0 | TTA vô hại nhưng vô ích; không đưa vào, khỏi thêm biến |

**Mốc đối chiếu E4** (WORKLOG S-078): fold 1 = 0.7001 · 2 = 0.6771 · 3 = 0.7304 ·
4 = 0.6680 · 5 = 0.6618 · gộp 394 ca = 0.6851.

⚠️ Con số từng fold ở dưới **chưa phải kết quả báo cáo**. Bootstrap ghép cặp trên đủ
394 ca chạy ở máy local sau khi tải kết quả về.

## Cần mount hai dataset

1. **cache E4** (nhận diện bằng `cache_meta.json`, không bằng tên)
2. **checkpoint** — `best_fold_1..5.pt` phẳng, hoặc `fold_N/best.pt`

## 0. Bootstrap

Dòng `repo commit` là bằng chứng đang chạy đúng bản code nào.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"

# ---- THAM SỐ ---------------------------------------------------------------
FOLDS = [1, 2, 3, 4, 5]
TTA_SET = "all"        # "all" = 8 lượt (lật 3 trục) · "inplane" = 4 lượt
# ----------------------------------------------------------------------------

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)

os.environ["LLDMMRI_OUTPUT_DIR"] = "/kaggle/working/runs/E4_tta"
os.environ.pop("LLDMMRI_DATA_ROOT", None)

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

CFG_PATH = REPO / "configs" / "baseline_3dpatch.yaml"
CFG = load_yaml(CFG_PATH)
print(f"config: {CFG_PATH.name} · fold {FOLDS} · TTA '{TTA_SET}'")

## 1. Cache E4 và checkpoint

Cần **hai** thứ mount vào: cache E4 (`lesion_tight · 112×112×32 · per_phase`) và 5 file
`best.pt`. Đổi đường dẫn bên dưới cho khớp tên dataset bạn đã upload.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")

# Cache E4 nhận diện bằng NỘI DUNG `cache_meta.json`, không bằng tên dataset. Tên do
# người upload đặt và đã lệch một lần rồi (`lld-mmri-lesion-tight/cache_lesion_tight`
# chứ không phải `lld-mmri-e4-per-phase` như đoán ở S-080). Ba khoá này là thứ phân
# biệt E4 với mọi cache trước đó.
E4_KEYS = {
    "align_phases": "per_phase",          # <- phân biệt E4 với E3
    "target_size": [112, 112, 32],        # <- phân biệt E3/E4 với E0/E1
    "crop_mode": "lesion_tight",          # <- phân biệt E1+ với E0
}

# Tên file checkpoint. KHÔNG kèm thư mục cha — độ sâu do `rglob` lo, xem bên dưới.
#   A) best_fold_1.pt ... best_fold_5.pt   <- dataset "best weights"
#   B) fold_1/best.pt ...                   <- gói thẳng từ output run
CKPT_NAMES = ["best_fold_{f}.pt", "best.pt"]

# ---------------------------------------------------------------------------
# KHÔNG hardcode độ sâu. Kaggle mount ở `/kaggle/input/datasets/<user>/<slug>/...`
# chứ không phải `/kaggle/input/<slug>/...` như mọi notebook trước giả định, và độ
# sâu đó có thể đổi tiếp. Dò theo TÊN FILE mốc, sâu bao nhiêu cũng thấy. Đây là lần
# thứ tư sửa cùng một lớp lỗi (S-081 → S-084); nguyên nhân gốc luôn là một giả định
# về hình dạng đường dẫn.
#
# MỘT lượt `os.walk` duy nhất thu hết mọi thứ cần. Không dùng nhiều `rglob` riêng:
# dataset gốc là 83.7GB / ~4000 file trên ổ mạng, và mỗi `rglob` là một lượt duyệt
# toàn cây — 11 lượt thì chờ rất lâu mà chẳng được gì thêm.
# ---------------------------------------------------------------------------
import json as _json
import os as _os
import re as _re

_cfg_data = load_yaml(REPO / "configs" / "data.yaml")
_ann_name = Path(_cfg_data["annotation_rel"]).name

interesting = {}       # thư mục -> số .npz/.pt/meta, để in bảng chẩn đoán
meta_paths = []        # cache_meta.json tìm được
ckpt_paths = []        # mọi file .pt tên best*.pt
_ann = []              # file annotation của dữ liệu gốc

for dirpath, dirnames, filenames in _os.walk(INPUT_ROOT):
    dirnames[:] = [x for x in dirnames if x not in (".cache", ".git")]  # rác tải HF
    here = {"npz": 0, "pt": 0, "meta": 0}
    for name in filenames:
        full = Path(dirpath) / name
        if name.endswith(".npz"):
            here["npz"] += 1
        elif name.endswith(".pt"):
            here["pt"] += 1
            if name.startswith("best"):
                ckpt_paths.append(full)
        elif name == "cache_meta.json":
            here["meta"] += 1
            meta_paths.append(full)
        elif name == _ann_name:
            _ann.append(full)
    if any(here.values()):
        interesting[Path(dirpath)] = here


def read_caches(paths):
    out = []
    for p in sorted(paths):
        try:
            out.append((p.parent, _json.loads(p.read_text("utf-8"))))
        except Exception as exc:  # noqa: BLE001 - chỉ để báo cáo, không nuốt lỗi thật
            out.append((p.parent, {"__loi__": repr(exc)}))
    return out


def matches_e4(meta):
    return all(meta.get(k) == v for k, v in E4_KEYS.items())


def pick_checkpoints(paths, folds):
    """{fold: đường dẫn}. Ưu tiên `best_fold_N.pt`; `best.pt` thì suy fold từ thư
    mục cha (`fold_3/best.pt`). Không suy được thì bỏ, không đoán bừa."""
    out = {}
    for fold in folds:
        hits = [p for p in sorted(paths) if p.name == f"best_fold_{fold}.pt"]
        if not hits:
            hits = [
                p for p in sorted(paths)
                if p.name == "best.pt"
                and (m := _re.search(r"fold_?(\d+)", p.parent.name))
                and int(m.group(1)) == fold
            ]
        if hits:
            out[fold] = hits[0]
    return out


print(f"=== Thư mục có dữ liệu dưới {INPUT_ROOT} ===")
for d in sorted(interesting)[:25]:
    c = interesting[d]
    print(f"  {d}\n      {' · '.join(f'{c[k]} {k}' for k in ('npz', 'pt', 'meta') if c[k])}")
if not interesting:
    print("  (trống — chưa mount dataset nào có .npz/.pt)")

print(f"\n=== Dữ liệu gốc ({_ann_name}) ===")
for p in sorted(_ann)[:5]:
    print(f"  ✓ {p.parent.parent}")
if not _ann:
    print("  KHÔNG thấy — chỉ cần nếu phải build cache (xem ngay dưới)")

caches = read_caches(meta_paths)
print(f"\n=== {len(caches)} cache có cache_meta.json ===")
for path, meta in caches:
    mark = "✓ E4" if matches_e4(meta) else "  --"
    print(
        f"  {mark}  {path}\n"
        f"        crop={meta.get('crop_mode')} size={meta.get('target_size')} "
        f"align={meta.get('align_phases')}"
    )

e4 = [p for p, m in caches if matches_e4(m)]
if e4:
    CACHE_DIR = e4[0]
    BUILD_NEEDED = False
else:
    # Thư mục có nhiều .npz nhưng KHÔNG có meta: không dùng được, và phải nói rõ vì sao.
    # Hình dạng mảng cho biết target_size, nhưng KHÔNG cho biết `align_phases` —
    # E3 (reference) và E4 (per_phase) có cùng shape [8,112,112,32]. Nhận nhầm E3
    # thành E4 sẽ cho ra một bảng kết quả sai mà trông hoàn toàn hợp lý.
    for d, c in sorted(interesting.items()):
        if c["npz"] > 100 and not c["meta"]:
            print(f"\n⚠ {d} có {c['npz']} file .npz nhưng KHÔNG có cache_meta.json.")
            print("  Không dùng được: shape cho biết target_size nhưng KHÔNG phân biệt được")
            print("  E3 (align=reference) với E4 (align=per_phase) — hai cái cùng shape.")
    BUILD_NEEDED = True
    CACHE_DIR = Path("/kaggle/working/cache_e4")
    if _ann:
        print("\n=> sẽ BUILD lại cache E4 (~26 phút). Dữ liệu gốc đã có ✓")
    else:
        print(
            "\n=> CẦN BUILD cache E4 nhưng CHƯA MOUNT dữ liệu gốc.\n"
            f"   Mount dataset chứa {_cfg_data['annotation_rel']} "
            f"(ứng viên: {_cfg_data.get('data_root_candidates')}),\n"
            "   rồi chạy lại từ cell này."
        )

CKPTS = pick_checkpoints(ckpt_paths, FOLDS)
thieu = [f for f in FOLDS if f not in CKPTS]
assert not thieu, (
    f"không thấy checkpoint cho fold {thieu}.\n"
    f"Đã dò theo tên {CKPT_NAMES} ở MỌI độ sâu dưới {INPUT_ROOT}.\n"
    f"Tìm được: { {f: str(p) for f, p in CKPTS.items()} }"
)
print("\ncache:      ", CACHE_DIR, "(CHƯA CÓ — sẽ build ở cell dưới)" if BUILD_NEEDED else "")

# 5 file cùng kiến trúc nên cùng kích thước — kích thước KHÔNG chứng minh chúng khác
# nhau. Băm để chắc không phải một file bị chép 5 lần với 5 cái tên.
import hashlib

print("checkpoint:")
digests = {}
for f in FOLDS:
    p = CKPTS[f]
    h = hashlib.sha256(p.read_bytes()).hexdigest()[:16]
    digests[f] = h
    print(f"  fold {f}: {p.name}  {p.stat().st_size / 2**20:.1f} MB  sha256 {h}")
assert len(set(digests.values())) == len(FOLDS), f"có checkpoint trùng nhau: {digests}"

# Đối chiếu với mã băm đo ở máy local (WORKLOG S-081). Khác => file khác bản.
LOCAL_SHA = {
    1: "2e1f3e1ad477ad59", 2: "30a8eb9ee221d453", 3: "00c133e031bdf8fe",
    4: "3fe18f1eb3de4431", 5: "d61cc7ed94b8ebf0",
}
lech = {f: (digests[f], LOCAL_SHA[f]) for f in FOLDS if f in LOCAL_SHA and digests[f] != LOCAL_SHA[f]}
if lech:
    print(f"\n⚠ mã băm khác bản local: {lech}")
    print("  Không tự dừng — nhưng nếu bạn không cố ý đổi checkpoint thì hãy dừng lại xem.")

## 1b. Build cache E4 nếu chưa có

Chỉ chạy khi cell trên không tìm thấy cache E4 nào. Build lại **cho ra đúng cùng dữ
liệu** — pipeline tiền xử lý tất định, `set_seed` chỉ ảnh hưởng train.

`resolve_data_root` tự lùng dataset LLD-MMRI gốc dưới `/kaggle/input` bằng cách tìm
file annotation, nên không cần khai đường dẫn. Nếu chưa mount dataset gốc thì cell
này sẽ báo rõ chứ không build ra cache rỗng.

In [ ]:
if BUILD_NEEDED:
    from src.utils.io import resolve_data_root

    # Data root khai ở configs/data.yaml, KHÔNG ở preprocess_*.yaml — file preprocess
    # chỉ có tham số tiền xử lý. `build_cache` cũng đọc data.yaml (xem hàm main của
    # nó). Kiểm trước ở đây chỉ để fail nhanh, thay vì chết giữa job 26 phút.
    cfg_data = load_yaml(REPO / "configs" / "data.yaml")
    try:
        data_root = resolve_data_root(cfg_data)
    except Exception as exc:
        # RuntimeError chứ không SystemExit: SystemExit làm IPython lỗi khi dựng
        # traceback và che mất thông báo thật bằng một trang lỗi của chính nó.
        data_root, exc_msg = None, str(exc)
    else:
        exc_msg = None

    # `resolve_data_root` trả về `config['data_root']` mà KHÔNG xác minh khi mọi cách
    # dò đều trượt (src/utils/io.py:219-225). Trên Kaggle nó sẽ là `data/lldmmridataset`
    # tương đối, không tồn tại — và job 26 phút sẽ chết giữa chừng. Xác minh ở đây.
    ann = (data_root / cfg_data["annotation_rel"]) if data_root else None
    if ann is None or not ann.exists():
        raise RuntimeError(
            f"Không tìm thấy dữ liệu LLD-MMRI gốc.\n"
            f"  resolve_data_root -> {data_root}"
            + (f" (lỗi: {exc_msg})" if exc_msg else f", nhưng {ann} không tồn tại")
            + f"\n  Cần mount dataset chứa {cfg_data['annotation_rel']}.\n"
            f"  Ứng viên khai trong configs/data.yaml: {cfg_data.get('data_root_candidates')}"
        ) from None
    print("data root:", data_root, "✓")

    os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)
    rc = subprocess.run(
        [sys.executable, "-m", "src.preprocess.build_cache",
         "--config", "configs/preprocess_e4.yaml"],
        cwd=REPO,
    ).returncode
    assert rc == 0, "build cache thất bại"
    print("build xong:", CACHE_DIR)
else:
    print("bỏ qua build — đã có cache E4:", CACHE_DIR)

os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)

## Cổng A ⚠️ — cache có đúng là E4 không

Chạy MC-dropout trên cache của E1 hay E3 sẽ **không báo lỗi gì cả**, chỉ lặng lẽ cho ra
số sai.

In [ ]:
import json

meta = json.loads((Path(os.environ["LLDMMRI_CACHE_DIR"]) / "cache_meta.json").read_text("utf-8"))

# Dùng lại E4_KEYS của cell trên, không chép ra bản thứ hai — hai bản sẽ trôi khỏi nhau.
for key, want in E4_KEYS.items():
    got = meta.get(key)
    assert got == want, f"cache SAI: {key} = {got!r}, cần {want!r}. Đây không phải cache E4."
assert meta["lesion_tight"]["source"] == "mask", "phải cắt theo mask, không phải bbox"

n_npz = len(list(Path(os.environ["LLDMMRI_CACHE_DIR"]).glob("*.npz")))
assert n_npz >= 498, f"chỉ có {n_npz} ca, cần 498 — cache chưa build xong"
print(f"cache_meta khớp E4 ✓ · {n_npz} ca · commit {meta.get('git_commit')}")

## Cổng B ⚠️⚠️ — model có ở chế độ eval không

Đây là cái bẫy đã đốt một buổi ở S-096: `build_model` trả về model ở **chế độ train**.
Quên `model.eval()` thì BatchNorm dùng thống kê của batch hiện tại và Dropout vẫn bật,
nên kết quả **đổi giữa hai lần chạy giống hệt nhau**. Với TTA thì hậu quả tệ hơn: 8
lượt sẽ khác nhau vì nhiễu chứ không phải vì phép lật, và phần "lợi ích" đo được chỉ là
trung bình cộng của nhiễu.

In [ ]:
import torch

from src.models import build_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

probe = build_model(CFG["model"])
assert probe.training, "giả định ở S-096 đã đổi — đọc lại build_model trước khi chạy tiếp"
probe.eval()
assert not any(m.training for m in probe.modules()), "còn module ở chế độ train"
print("build_model trả về chế độ train; .eval() tắt được toàn bộ ✓")
del probe

## 2. Chạy TTA từng fold

Mỗi fold: nạp checkpoint, chạy 8 lượt, so lượt 0 (ảnh gốc) với trung bình 8 lượt.

`assert` ở cuối vòng lặp là phần quan trọng nhất của cell này. Nó đối chiếu lượt 0 với
macro-F1 **đã lưu trong chính checkpoint** lúc train, nên không phải số tôi gõ tay vào
đây. Lệch quá 0.002 nghĩa là có gì đó khác với lúc train — gần như luôn là sai cache
hoặc sai fold — và mọi con số TTA phía sau sẽ vô nghĩa.

In [ ]:
import time

import numpy as np

from src.eval.metrics import macro_f1
from src.eval.tta import FLIP_SETS, flip_combinations, tta_predict
from src.train.run import build_loaders

OUT_ROOT = Path(os.environ["LLDMMRI_OUTPUT_DIR"])
AXES = FLIP_SETS[TTA_SET]
print(f"TTA '{TTA_SET}': trục {AXES} -> {len(flip_combinations(AXES))} lượt/ca\n")

print(f"{'fold':>5}{'n':>5}{'lưu':>9}{'lượt 0':>9}{'TTA':>9}{'hiệu':>9}{'giây':>7}")
print("-" * 53)
gain, rows = [], []
for fold in FOLDS:
    t0 = time.time()
    _, val_loader, _ = build_loaders(CFG, fold)

    state = torch.load(CKPTS[fold], map_location=DEVICE)
    if state.get("fold") not in (None, fold):
        raise RuntimeError(f"checkpoint ghi fold={state['fold']} nhưng đang chạy fold {fold}")

    model = build_model(CFG["model"]).to(DEVICE)
    model.load_state_dict(state["model"])
    model.eval()          # BẮT BUỘC — xem Cổng B
    assert not any(m.training for m in model.modules()), "còn module ở chế độ train"

    out = tta_predict(model, val_loader, DEVICE, axes=AXES,
                      amp=bool(CFG["train"].get("amp", True)))

    base = macro_f1(out["labels"], out["probs_per_view"][0].argmax(1))
    with_tta = macro_f1(out["labels"], out["probs"].argmax(1))
    saved = (state.get("metrics") or {}).get("macro_f1")
    gain.append(with_tta - base)
    rows.append((fold, saved, base, with_tta))
    print(
        f"{fold:>5}{len(out['labels']):>5}"
        f"{(f'{saved:.4f}' if saved is not None else '   —'):>9}"
        f"{base:>9.4f}{with_tta:>9.4f}{with_tta - base:>+9.4f}{time.time() - t0:>7.0f}"
    )

    d = OUT_ROOT / f"fold_{fold}"
    d.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        d / "val_probs_best_tta.npz",
        probs=out["probs"],
        probs_per_view=out["probs_per_view"].astype(np.float16),
        labels=out["labels"],
        patient_ids=np.array(out["patient_ids"]),
        # views dạng CHUỖI, không phải object array: `src.eval.run.load_predictions`
        # mở file với `allow_pickle=False`, mà object array thì cần pickle.
        views=np.array(["-".join(map(str, v)) or "goc" for v in out["views"]]),
        n_views=len(out["views"]),
        epoch=int(state.get("epoch", -1)),
    )
    # Lượt 0 là ảnh gốc, nên nó PHẢI dựng lại đúng con số đã lưu lúc train. Đây là
    # thứ duy nhất chứng minh cache/checkpoint/chế độ eval đều đúng.
    if saved is not None:
        assert abs(base - saved) < 2e-3, (
            f"fold {fold}: lượt 0 cho {base:.4f} nhưng checkpoint lưu {saved:.4f}. "
            f"Không phải chuyện TTA — sai cache, sai fold, hoặc model còn ở train mode. "
            f"DỪNG, đừng đọc các số TTA."
        )
    del model
    torch.cuda.empty_cache()

print(f"\ntrung bình qua {len(gain)} fold: {np.mean(gain):+.4f}"
      f"  ·  dương ở {sum(g > 0 for g in gain)}/{len(gain)} fold")
print(
    "\n⚠ Đây là hiệu TRÊN TỪNG FOLD, chưa phải con số báo cáo. Trung bình các fold\n"
    "  KHÔNG có CI đúng nghĩa (mỗi fold là một tập nhỏ khác nhau). Gộp out-of-fold và\n"
    "  bootstrap ghép cặp làm ở máy local sau khi tải kết quả về."
)

## 3. Latency suy luận — đo trên val, không chạm test

Lần chạm test-104 (WORKLOG S-110) có sẵn con số này nhưng code lúc đó không ghi lại,
và test chạm đúng một lần nên không chạy lại để đo được.

**Đo trên val cho ra đúng con số đó.** Cùng checkpoint, cùng khối đầu vào
`[8, 112, 112, 32]`, cùng đường code. Latency không phụ thuộc ca nào nằm trong tập,
nên không có lý do gì phải chạm test để lấy nó.

Ba điều khiến phép đo này đúng thay vì chỉ trông có vẻ đúng:

1. **Bấm giờ sau khi nạp checkpoint**, để đo suy luận chứ không đo I/O đọc file `.pt`.
2. **`torch.cuda.synchronize()` hai đầu.** Lệnh CUDA chạy bất đồng bộ; thiếu nó thì
   đồng hồ dừng lúc hàng đợi được xếp xong, không phải lúc GPU tính xong. Đây là cách
   dễ nhất để báo một con số nhanh gấp nhiều lần sự thật.
3. **Bỏ lượt đầu (warm-up).** Lần forward đầu tiên gánh chi phí khởi tạo cuDNN và cấp
   phát bộ nhớ, thường chậm hơn hẳn phần còn lại.

In [ ]:
import time

import numpy as np
import torch

from src.models import build_model
from src.train.run import build_loaders

LAT_FOLD = FOLDS[0] if "FOLDS" in dir() else 1

_, lat_loader, _ = build_loaders(CFG, LAT_FOLD)
state = torch.load(CKPTS[LAT_FOLD], map_location=DEVICE)
model = build_model(CFG["model"]).to(DEVICE)
model.load_state_dict(state["model"])
model.eval()

amp = bool(CFG["train"].get("amp", True))
n_cases = len(lat_loader.dataset)


def _one_pass():
    """Một lượt forward trọn tập val. Trả về số giây GPU thật sự tốn."""
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for batch in lat_loader:
            images = batch["image"].to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=amp):
                model(images)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    return time.perf_counter() - t0


_one_pass()                                    # warm-up, bỏ đi
runs = [_one_pass() for _ in range(3)]
per_case_ms = np.array(runs) / n_cases * 1000

print(f"fold {LAT_FOLD} · {n_cases} ca · batch {lat_loader.batch_size} · amp={amp} · {DEVICE}")
print(f"ba lượt: {', '.join(f'{r:.2f}s' for r in runs)}")
print()
print(f"  1 model        : {per_case_ms.mean():6.1f} ms/ca  (±{per_case_ms.std():.1f})")
print(f"  ensemble 5 fold: {per_case_ms.mean() * 5:6.1f} ms/ca")
print()
print("⚠ Đây CHỈ là phần model, và đọc từ cache đã tiền xử lý sẵn.")
print("  Một ca MỚI còn phải tiền xử lý trước: trung vị 3,43s, p90 4,74s trên CPU")
print("  (đo thật trên 498 ca, cache_build_log.csv). Tiền xử lý mới là phần chiếm")
print("  gần hết thời gian chờ của người dùng, không phải model.")

del model
torch.cuda.empty_cache()

## 4. Gói mang về

Chỉ `val_probs_best_tta.npz`, mỗi fold vài trăm KiB. Giải nén về
`runs/E4_per_phase_results/fold_N/` để nằm cạnh `val_probs_best.npz` sẵn có, rồi so cặp
ở local.

In [ ]:
import shutil

PACK = Path("/kaggle/working/E4_tta_results")
shutil.rmtree(PACK, ignore_errors=True)
PACK.mkdir(parents=True)

for d in sorted(OUT_ROOT.glob("fold*")):
    src = d / "val_probs_best_tta.npz"
    if src.exists():
        (PACK / d.name).mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, PACK / d.name / src.name)

total = sum(f.stat().st_size for f in PACK.rglob("*") if f.is_file())
print(f"đã gói {PACK}: {total / 2**20:.2f} MiB")
for f in sorted(PACK.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(PACK)}  {f.stat().st_size / 2**10:.0f} KiB")

print("""
⚠ TẢI VỀ: giải nén CHỈ MỘT LỚP. File .npz bản thân là zip; trình giải nén bung đệ quy
  sẽ biến nó thành thư mục và src.eval.* sẽ không thấy (đã dính hai lần, S-078).

Đặt vào runs/E4_per_phase_results/fold_N/val_probs_best_tta.npz rồi chạy ở local:
    python -m src.eval.run --run-dir runs/E4_per_phase_results
""")